# QuantAlpha AI — Step 3: Mode-Aware Ensemble Scoring Engine + Backtest

Combines everything built so far:
- **Step 1:** Technical indicators (`data/technical/*.parquet`)
- **Step 2 Part A:** Fundamental scores — Piotroski F-Score, ROCE, FCF margin (`data/fundamental_scores.csv`)
- **Step 2B:** Event classification — temporary vs structural news events (`data/event_classification.csv`)

**Output:** a single 0-100 score per stock per mode (Swing / Position / Long-Term), ranked into
Top 10/25/50 lists — plus a backtest so you get a REAL win-rate number, not a guess.

Run top to bottom. This is the notebook that finally answers "does this actually work."


## 1. Connect to Drive + load all saved data

In [1]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.chdir('/content/drive/MyDrive/QuantAlpha')

import pandas as pd
import numpy as np
import glob
import warnings
warnings.filterwarnings('ignore')

fundamental_scores = pd.read_csv('data/fundamental_scores.csv')
event_classification = pd.read_csv('data/event_classification.csv')

print(f"Fundamental scores loaded: {len(fundamental_scores)} stocks")
print(f"Event classifications loaded: {len(event_classification)} events")


Mounted at /content/drive
Fundamental scores loaded: 200 stocks
Event classifications loaded: 92 events


## 2. Load technical snapshot per stock
We only need the LATEST row per stock for scoring (today's indicator values), not full history —
full history gets used later in the backtest section.


In [2]:
technical_files = glob.glob('data/technical/*.parquet')
symbols = sorted(f.split('/')[-1].replace('.parquet', '') for f in technical_files)
print(f"Found {len(symbols)} stocks with technical data")

def get_latest_technical(symbol):
    df = pd.read_parquet(f"data/technical/{symbol}.parquet")
    if df.empty:
        return None
    latest = df.iloc[-1].to_dict()
    latest['symbol_short'] = symbol
    return latest

technical_latest = []
for sym in symbols:
    row = get_latest_technical(sym)
    if row:
        technical_latest.append(row)

technical_df = pd.DataFrame(technical_latest)
print(f"Latest technical snapshot built for {len(technical_df)} stocks")
technical_df[['symbol_short', 'Close', 'RSI_14', 'ADX_14']].head()


Found 200 stocks with technical data
Latest technical snapshot built for 200 stocks


,symbol_short,Close,RSI_14,ADX_14
0,360ONE,1109.500000,54.230963,12.984803
1,ABB,6950.500000,48.387729,13.099004
2,ABCAPITAL,401.250000,66.830169,28.670621
3,ADANIENSOL,1568.800049,63.236507,21.079490
4,ADANIENT,3212.100098,69.244024,38.935018


## 3. Normalize symbol formats and merge all 3 datasets
Step 1 saved symbols without `.NS` suffix in filenames, Step 2 kept `.NS`. Normalize both to the
same key before merging.


In [3]:
fundamental_scores['symbol_short'] = fundamental_scores['symbol'].str.replace('.NS', '', regex=False)
event_classification['symbol_short'] = event_classification['symbol'].str.replace('.NS', '', regex=False) if len(event_classification) else pd.Series(dtype=str)

# Aggregate events per stock: most recent verdict + whether ANY structural risk flag exists
if len(event_classification):
    event_summary = event_classification.groupby('symbol_short').agg(
        latest_verdict=('verdict', 'first'),
        has_structural_flag=('verdict', lambda x: any('STRUCTURAL' in v for v in x)),
        negative_news_count=('headline', 'count')
    ).reset_index()
else:
    event_summary = pd.DataFrame(columns=['symbol_short', 'latest_verdict', 'has_structural_flag', 'negative_news_count'])

master = technical_df.merge(fundamental_scores, on='symbol_short', how='left')
master = master.merge(event_summary, on='symbol_short', how='left')
master['has_structural_flag'] = master['has_structural_flag'].fillna(False)
master['negative_news_count'] = master['negative_news_count'].fillna(0)

print(f"Master merged dataset: {len(master)} stocks, {master.shape[1]} columns")
master[['symbol_short', 'piotroski_f_score', 'roce', 'has_structural_flag']].head(10)


Master merged dataset: 200 stocks, 46 columns


,symbol_short,piotroski_f_score,roce,has_structural_flag
0,360ONE,3,0.171741,False
1,ABB,5,0.176629,True
2,ABCAPITAL,4,NaN,False
3,ADANIENSOL,7,0.198309,False
4,ADANIENT,4,0.113337,False
5,ADANIGREEN,6,0.188282,False
6,ADANIPORTS,6,0.218880,False
7,ADANIPOWER,6,0.264774,False
8,ALKEM,6,0.187572,False
9,AMBUJACEM,8,0.086664,False


## 4. Scoring functions per engine (0-100 scale)
Each engine outputs a 0-100 score using percentile ranking WITHIN the current universe — so a
stock is scored relative to its Nifty 200 peers, not against an arbitrary fixed threshold.


In [4]:
def percentile_score(series):
    """Convert raw values to a 0-100 score based on rank within the universe.
    Higher raw value = higher score. NaNs get a neutral 50."""
    ranked = series.rank(pct=True, na_option='keep') * 100
    return ranked.fillna(50)

# --- Fundamental Score ---
master['fundamental_score'] = (
    0.35 * percentile_score(master['piotroski_f_score']) +
    0.25 * percentile_score(master['roce']) +
    0.20 * percentile_score(master['revenue_growth']) +
    0.20 * percentile_score(master['net_income_growth'])
)

# --- Cash Flow Score ---
master['cashflow_score'] = (
    0.5 * percentile_score(master['ocf_margin']) +
    0.5 * percentile_score(master['fcf_margin'])
)

# --- Technical Score ---
# RSI: prefer 40-65 zone (momentum without being overbought) -- distance-based scoring
def rsi_zone_score(rsi):
    if pd.isna(rsi):
        return 50
    if 40 <= rsi <= 65:
        return 100 - abs(rsi - 52.5) * 2  # peak near 52.5
    elif rsi > 65:
        return max(0, 100 - (rsi - 65) * 3)  # penalize overbought
    else:
        return max(0, 60 - (40 - rsi) * 2)  # penalize oversold/weak

master['rsi_component'] = master['RSI_14'].apply(rsi_zone_score)
master['trend_component'] = percentile_score(master['ADX_14'])  # higher ADX = stronger trend
master['momentum_component'] = percentile_score(master['pct_from_52w_low'])
master['technical_score'] = (
    0.4 * master['rsi_component'] +
    0.3 * master['trend_component'] +
    0.3 * master['momentum_component']
)

# --- Sentiment/Event Score ---
def event_score(row):
    if row['negative_news_count'] == 0:
        return 70  # neutral-positive: no negative news flagged
    if row['has_structural_flag']:
        return 20  # real risk flagged
    return 55  # negative news but classified as temporary

master['sentiment_score'] = master.apply(event_score, axis=1)

print("Engine scores computed.")
master[['symbol_short', 'fundamental_score', 'cashflow_score', 'technical_score', 'sentiment_score']].describe()


Engine scores computed.


,fundamental_score,cashflow_score,technical_score,sentiment_score
count,200.000000,200.000000,200.000000,200.000000
mean,50.250000,50.250000,62.652191,57.975000
std,19.331634,27.156654,14.519359,19.008709
min,6.951953,0.510204,26.088821,20.000000
25%,35.086105,27.742347,51.499569,55.000000
50%,49.629851,51.275510,62.216485,70.000000
75%,65.239023,72.257653,72.983460,70.000000
max,95.345716,100.000000,95.299211,70.000000


## 5. Mode-aware weighting
Engine importance shifts by investment mode — matches Section 5 of your project doc.


In [5]:
MODE_WEIGHTS = {
    'swing': {
        'fundamental_score': 0.10,
        'cashflow_score': 0.05,
        'technical_score': 0.60,
        'sentiment_score': 0.25
    },
    'position': {
        'fundamental_score': 0.30,
        'cashflow_score': 0.15,
        'technical_score': 0.35,
        'sentiment_score': 0.20
    },
    'longterm': {
        'fundamental_score': 0.45,
        'cashflow_score': 0.25,
        'technical_score': 0.15,
        'sentiment_score': 0.15
    }
}

for mode, weights in MODE_WEIGHTS.items():
    master[f'{mode}_overall_score'] = sum(master[engine] * w for engine, w in weights.items())

print("Mode-aware overall scores computed.")
master[['symbol_short', 'swing_overall_score', 'position_overall_score', 'longterm_overall_score']].head(10)


Mode-aware overall scores computed.


,symbol_short,swing_overall_score,position_overall_score,longterm_overall_score
0,360ONE,53.953381,44.965560,36.826940
1,ABB,48.741871,41.105848,35.420769
2,ABCAPITAL,74.427558,56.452167,41.161521
3,ADANIENSOL,72.671689,68.066915,64.059463
4,ADANIENT,75.449243,55.017330,38.030449
5,ADANIGREEN,74.204647,63.162793,54.361988
6,ADANIPORTS,66.412663,65.969889,66.005792
7,ADANIPOWER,64.579194,56.707664,50.425284
8,ALKEM,65.926033,58.605218,52.685512
9,AMBUJACEM,50.167382,50.012967,49.348735


## 6. Ranked Top 10/25/50 lists per mode
This is your actual deliverable — the ranked recommendation lists.


In [6]:
def get_top_n(mode, n=10):
    col = f'{mode}_overall_score'
    top = master.nlargest(n, col)[[
        'symbol_short', col, 'fundamental_score', 'technical_score',
        'cashflow_score', 'sentiment_score', 'has_structural_flag'
    ]].copy()
    top = top.rename(columns={col: 'overall_score'})
    top['rank'] = range(1, len(top) + 1)
    return top[['rank', 'symbol_short', 'overall_score', 'fundamental_score',
                'technical_score', 'cashflow_score', 'sentiment_score', 'has_structural_flag']]

print("=== TOP 10 — SWING TRADING ===")
print(get_top_n('swing', 10).to_string(index=False))


=== TOP 10 — SWING TRADING ===
 rank symbol_short  overall_score  fundamental_score  technical_score  cashflow_score  sentiment_score  has_structural_flag
    1       RADICO      84.198831          84.286682        92.992788       49.489796               70                False
    2   LAURUSLABS      83.405933          83.185838        90.494221       65.816327               70                False
    3         IDEA      83.289553          68.064040        91.162392       85.714286               70                False
    4        NYKAA      82.046980          82.206442        90.858518       36.224490               70                False
    5   FEDERALBNK      80.050308          49.626187        95.299211       83.163265               55                False
    6   GMRAIRPORT      79.919143          95.345716        81.805919       76.020408               70                False
    7      YESBANK      79.029153          66.491106        83.175920       99.489796               7

In [7]:
print("=== TOP 10 — POSITION TRADING ===")
print(get_top_n('position', 10).to_string(index=False))


=== TOP 10 — POSITION TRADING ===
 rank symbol_short  overall_score  fundamental_score  technical_score  cashflow_score  sentiment_score  has_structural_flag
    1          MCX      83.920052          93.348989        76.901014      100.000000               70                False
    2   GMRAIRPORT      82.638848          95.345716        81.805919       76.020408               70                False
    3   LAURUSLABS      80.501178          83.185838        90.494221       65.816327               70                False
    4       RADICO      79.256950          84.286682        92.992788       49.489796               70                False
    5         IDEA      79.183192          68.064040        91.162392       85.714286               70                False
    6          BSE      78.920811          90.199343        66.519789       97.193878               70                False
    7      YESBANK      77.982373          66.491106        83.175920       99.489796             

In [8]:
print("=== TOP 10 — LONG-TERM INVESTING ===")
print(get_top_n('longterm', 10).to_string(index=False))


=== TOP 10 — LONG-TERM INVESTING ===
 rank symbol_short  overall_score  fundamental_score  technical_score  cashflow_score  sentiment_score  has_structural_flag
    1          MCX      89.042197          93.348989        76.901014      100.000000               70                False
    2          BSE      85.366142          90.199343        66.519789       97.193878               70                False
    3   GMRAIRPORT      84.681562          95.345716        81.805919       76.020408               70                False
    4        ENRIN      81.137269          84.100244        64.277663       92.602041               70                False
    5   NATIONALUM      80.199521          93.834220        86.051971       88.265306               20                 True
    6     HINDZINC      80.176248          81.582341        69.251092       90.306122               70                False
    7        LUPIN      79.622883          84.158334        72.289798       81.632653          

In [9]:
# Save full ranked lists (Top 50 each) for all 3 modes
for mode in ['swing', 'position', 'longterm']:
    get_top_n(mode, 50).to_csv(f'data/rankings_{mode}.csv', index=False)

master.to_csv('data/master_scored_universe.csv', index=False)
print("Saved: data/rankings_swing.csv, data/rankings_position.csv, data/rankings_longterm.csv")
print("Saved: data/master_scored_universe.csv (full scored universe, all engines)")


Saved: data/rankings_swing.csv, data/rankings_position.csv, data/rankings_longterm.csv
Saved: data/master_scored_universe.csv (full scored universe, all engines)


## 7. Backtest — get your REAL accuracy number
**This is the most important cell in the whole project.** Everything above is a scoring opinion
until it's tested against what actually happened historically. This backtest checks: if you had
followed the Swing-mode Top 20 picks (rebalanced periodically using point-in-time technical data),
what would your actual forward return and win rate have been?

**Important limitation to be upfront about:** this is a SIMPLIFIED backtest using only current
technical indicators as a stand-in signal (since Steps 1-2 captured a snapshot, not full historical
fundamental/sentiment time series). It answers "does high technical + fundamental score correlate
with future returns historically" — a real production backtest would need historical point-in-time
scores for every past date, which is a larger future project. Treat this as a first directional
signal check, not your final validated accuracy.


In [10]:
def backtest_technical_signal(symbol, holding_days=10):
    """For each historical date, check: did a 'good technical setup' (RSI 40-65 + high ADX)
    predict a positive return over the next `holding_days` trading days?"""
    df = pd.read_parquet(f"data/technical/{symbol}.parquet")
    if len(df) < 300:
        return None

    df = df.reset_index(drop=True)
    signals = []
    for i in range(200, len(df) - holding_days):  # skip first 200 rows (indicator warmup)
        rsi = df.loc[i, 'RSI_14']
        adx = df.loc[i, 'ADX_14']
        if pd.isna(rsi) or pd.isna(adx):
            continue
        good_setup = (40 <= rsi <= 65) and (adx > 25)
        entry_price = df.loc[i, 'Close']
        exit_price = df.loc[i + holding_days, 'Close']
        fwd_return = (exit_price - entry_price) / entry_price
        signals.append({'good_setup': good_setup, 'fwd_return': fwd_return})

    return pd.DataFrame(signals)

all_backtest_results = []
for sym in symbols[:200]:
    result = backtest_technical_signal(sym, holding_days=10)
    if result is not None and len(result):
        all_backtest_results.append(result)

bt_df = pd.concat(all_backtest_results, ignore_index=True)
print(f"Total historical signal observations: {len(bt_df)}")


Total historical signal observations: 101279


In [11]:
good_setups = bt_df[bt_df['good_setup'] == True]
other_periods = bt_df[bt_df['good_setup'] == False]

win_rate_good = (good_setups['fwd_return'] > 0).mean() * 100
win_rate_baseline = (other_periods['fwd_return'] > 0).mean() * 100
avg_return_good = good_setups['fwd_return'].mean() * 100
avg_return_baseline = other_periods['fwd_return'].mean() * 100

print("=== BACKTEST RESULTS (10-day holding period, technical signal only) ===")
print(f"\n'Good setup' signal (RSI 40-65 + ADX>25):")
print(f"  Win rate: {win_rate_good:.1f}%  |  Avg 10-day return: {avg_return_good:+.2f}%  |  N={len(good_setups)}")
print(f"\nBaseline (all other periods):")
print(f"  Win rate: {win_rate_baseline:.1f}%  |  Avg 10-day return: {avg_return_baseline:+.2f}%  |  N={len(other_periods)}")
print(f"\nEdge over baseline: {win_rate_good - win_rate_baseline:+.1f} percentage points win rate")


=== BACKTEST RESULTS (10-day holding period, technical signal only) ===

'Good setup' signal (RSI 40-65 + ADX>25):
  Win rate: 52.1%  |  Avg 10-day return: +0.57%  |  N=18755

Baseline (all other periods):
  Win rate: 51.4%  |  Avg 10-day return: +0.50%  |  N=82524

Edge over baseline: +0.7 percentage points win rate


**How to read this honestly:**
- If the "good setup" win rate is meaningfully above the baseline (e.g. 55%+ vs 50%), the technical
  signal has SOME real edge — in line with the 55-62% range we discussed as realistic
- If it's roughly equal to baseline, the signal isn't adding value as currently defined, and the
  scoring weights/logic need revisiting before trusting it
- This tests ONLY the technical component in isolation — fundamental and sentiment scores aren't
  backtested here yet, since that needs historical point-in-time fundamental snapshots (a bigger
  future data engineering task, not a quick addition)


## 8. Summary
Saved to `data/`:
- `rankings_swing.csv`, `rankings_position.csv`, `rankings_longterm.csv` — Top 50 ranked picks per mode
- `master_scored_universe.csv` — every stock, every engine score, all 3 mode overall scores
- Backtest result printed above — your first real, honest accuracy signal

**What this notebook does NOT yet do (be aware):**
- Full historical backtesting of the COMBINED score (fundamental + technical + sentiment together)
- The "similar historical cases" feature from your original doc
- Explainable AI text generation (the "BUY — here's why" narrative)
- Entry zone / stop-loss / target price calculation

**Next (Step 4):** build the Explainable AI narrative generator (turns scores into the
"Recommendation: BUY, Confidence: 92%, Reasons: ..." format from your doc) and add
entry/target/stop-loss calculation using ATR-based risk levels.
